In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="0"

# LLM Inference Workshop – Part 1
## Tokenization, Chat Templates, and Basic Generation

In this notebook we will:

1. Load a causal language model.
2. Understand tokenization.
3. Apply a chat template.
4. Generate text using `.generate()`.
5. Experiment with sampling parameters.

This notebook focuses on **model I/O**, not serving optimization.

In [ ]:
from IPython.display import Markdown, display
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer

In [3]:
torch.manual_seed(0)

In [ ]:
if torch.backends.mps.is_available():
    device = "mps"
    dtype = torch.bfloat16   # float32 preferred on Apple Silicon
elif torch.cuda.is_available():
    device = "cuda"
    dtype = "auto"   # or bfloat16
else:
    device = "cpu"
    dtype = torch.float32 # Depends on the CPU type but FP32 is universal

print("Device:", device)
print("Dtype:", dtype)

Device: cuda
Dtype: auto


## Loading a Causal Language Model

We load:

- A tokenizer → converts text to token IDs
- A causal LM → predicts next tokens autoregressively

We use a relatively small instruct model for interactive experimentation.

In [5]:
# Choose a small instruct model to keep it lightweight
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load model in half precision if GPU is available
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=dtype,
).to(device=device)

# Set the model in evaluation mode
model.eval()

print("Model loaded.")

Model loaded.


## What is Tokenization?

LLMs do not operate on raw text.

They operate on **token IDs**.

The tokenizer:
- Splits text into subword tokens
- Maps each token to an integer ID
- Produces tensors usable by the model

Let’s inspect this process.

In [6]:
text = "Large Language Models are fascinating. Who are you agin?"

# Convert text to token IDs
encoded = tokenizer(text, return_tensors="pt", add_special_tokens=True)

print("Token IDs:")
print(encoded["input_ids"])

print("\nDecoded back with special tokens:")
print(tokenizer.decode(encoded["input_ids"][0], skip_special_tokens=False))

print("\nDecoded back without special tokens:")
print(tokenizer.decode(encoded["input_ids"][0], skip_special_tokens=True))

Token IDs:
tensor([[34253, 11434, 26874,   525, 26291,    13, 10479,   525,   498,   933,
           258,    30]])

Decoded back with special tokens:
Large Language Models are fascinating. Who are you agin?

Decoded back without special tokens:
Large Language Models are fascinating. Who are you agin?


In [7]:
tokenizer(text, add_special_tokens=True)

{'input_ids': [34253, 11434, 26874, 525, 26291, 13, 10479, 525, 498, 933, 258, 30], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [8]:
tokenizer(
    text,
    #return_tensors="pt",
    add_special_tokens=True
    )

{'input_ids': [34253, 11434, 26874, 525, 26291, 13, 10479, 525, 498, 933, 258, 30], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

#### JUST for comparison I also show another one:

In [ ]:
tokenizer2 = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")

# Convert text to token IDs
encoded2 = tokenizer2(text, return_tensors="pt", add_special_tokens=True)

print("Token IDs:")
print(encoded2["input_ids"])

print("\nDecoded back with special tokens:")
print(tokenizer2.decode(encoded2["input_ids"][0], skip_special_tokens=False))

print("\nDecoded back without special tokens:")
print(tokenizer2.decode(encoded2["input_ids"][0], skip_special_tokens=True))

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

Token IDs:
tensor([[128000,  35353,  11688,  27972,    527,  27387,     13,  10699,    527,
            499,    945,    258,     30]])

Decoded back with special tokens:
<|begin_of_text|>Large Language Models are fascinating. Who are you agin?

Decoded back without special tokens:
Large Language Models are fascinating. Who are you agin?


## Why Chat Templates?

Instruction-tuned models expect structured input.

Instead of raw text, they expect something like:

[System]
[User]
[Assistant]

A chat template formats conversation into the structure
the model was trained on.

If we skip this step, model behavior may degrade.

In [10]:
has_chat_template = lambda t:getattr(t, "chat_template", None) is not None

print(has_chat_template(tokenizer))
print(has_chat_template(tokenizer2))

True
False


In [11]:
# Create a simple conversation
messages = [
    {"role": "system", "content": "You are a helpful AI assistant."},
    # 
    {"role": "user", "content": "Explain PagedAttention in simple terms."}
    #
]

# Apply chat template
formatted_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True  # Adds assistant prefix
)

print("Formatted prompt:\n")
print(formatted_prompt)

Formatted prompt:

<|im_start|>system
You are a helpful AI assistant.<|im_end|>
<|im_start|>user
Explain PagedAttention in simple terms.<|im_end|>
<|im_start|>assistant



Now we convert the formatted prompt into token IDs
and move it to the correct device.

In [12]:
inputs = tokenizer(
    formatted_prompt,
    return_tensors="pt"
).to(device)

print("Input shape:", inputs["input_ids"].shape)

Input shape: torch.Size([1, 29])


## Autoregressive Generation (Greedy)

`.generate()` performs autoregressive decoding internally.

At each step:
1. Forward pass
2. Select next token
3. Append token
4. Repeat

By default, `model.generate()` uses a dynamically growing KV cache.

Here we instead:

- Preallocate the maximum KV memory upfront.
- Keep tensor shapes fixed.
- Avoid repeated tensor resizing.
- Make the model compatible with `torch.compile` and CUDA graphs.

Static caching reduces allocator overhead and often improves decode speed.

In [14]:
# Define how many tokens we plan to generate
max_new_tokens = 1024

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,  # Greedy decoding
        use_cache=True,
    )

response = tokenizer.decode(output_ids[0], skip_special_tokens=False)
print(response)

<|im_start|>system
You are a helpful AI assistant.<|im_end|>
<|im_start|>user
Explain PagedAttention in simple terms.<|im_end|>
<|im_start|>assistant
PagedAttention is an attention mechanism that allows for dynamic and flexible attention allocation across different parts of the input sequence, rather than being fixed to a single position or window.

In traditional attention mechanisms like Self-Attention, the focus is on aligning the query (input) with the key (key-value pair). However, this can lead to issues such as "vanilla" attention where the model only looks at one part of the input at a time, which may not be optimal for certain tasks.

PagedAttention addresses these limitations by allowing the model to dynamically allocate attention weights based on the current position within the input sequence. This means that instead of focusing solely on the first few tokens, it can consider all of them simultaneously, leading to more efficient processing and potentially better performance.

In [15]:
Markdown(tokenizer.decode(output_ids[0][len(inputs["input_ids"][0]) :], skip_special_tokens=True))

PagedAttention is an attention mechanism that allows for dynamic and flexible attention allocation across different parts of the input sequence, rather than being fixed to a single position or window.

In traditional attention mechanisms like Self-Attention, the focus is on aligning the query (input) with the key (key-value pair). However, this can lead to issues such as "vanilla" attention where the model only looks at one part of the input at a time, which may not be optimal for certain tasks.

PagedAttention addresses these limitations by allowing the model to dynamically allocate attention weights based on the current position within the input sequence. This means that instead of focusing solely on the first few tokens, it can consider all of them simultaneously, leading to more efficient processing and potentially better performance.

The key idea behind PagedAttention is to use a sliding window approach to look at each token in the sequence, considering its context from both sides. The attention weight is then adjusted accordingly based on how much information is available from each side of the token.

This method helps to:

1. Improve efficiency: It reduces the number of computations needed to process each token.
2. Enhance generalization: It allows the model to learn more about the entire sequence, rather than just a small portion.
3. Better handling of long sequences: It can handle longer sequences without becoming too slow or inefficient.

Overall, PagedAttention provides a powerful tool for improving the performance of models in various natural language processing tasks, particularly those involving sequential data.

## Sampling Strategies

Greedy decoding always picks the highest-probability token.

Sampling introduces randomness.

Common parameters:

- temperature → scales logits
- top_k → restrict to top K tokens
- top_p → nucleus sampling
- do_sample=True → enables stochastic decoding

In [16]:
# The default generation config
model.generation_config

GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": [
    151645,
    151643
  ],
  "pad_token_id": 151643,
  "repetition_penalty": 1.1,
  "temperature": 0.7,
  "top_k": 20,
  "top_p": 0.8
}

In [17]:
with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=1.1,
    )


Markdown(tokenizer.decode(output_ids[0][len(inputs["input_ids"][0]) :], skip_special_tokens=True))

PagedAttention is an attention mechanism that allows models to handle large amounts of sequential data, especially for tasks like image captioning or speech recognition.

In traditional attention mechanisms, the model uses the entire input sequence as context and looks at each token independently for relevance determination. However, this approach can lead to performance degradation due to overfitting.

PagedAttention addresses this issue by using memory tokens instead of the entire sequence. When an input sequence reaches its "persistence time," it stores only a portion of the token's information (called "paging"). These stored tokens serve as context for subsequent queries to the model, allowing it to consider only relevant information without overfitting.

The key idea behind pagedattention is to:

1. Store only relevant parts of a token rather than the entire sequence.
2. Maintain contextual information from previous queries.
3. Apply this contextual information selectively during inference.
4. Allow the model to focus on more specific areas of interest while still considering broader features.

This makes pagedAttention particularly useful for tasks with limited training data and require fast responses, such as image captioning or audio recognition.

To summarize:
- It replaces the entire sequence with smaller, more selective portions called "memory tokens."
- Allows the model to learn from past contexts efficiently, preventing overfitting.
- Enables faster inference by focusing on pertinent information rather than all of it simultaneously.

By doing so, pagedAttention enables models to perform well even when dealing with extremely large datasets and requiring quick updates or outputs.

In [18]:
with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
    )


Markdown(tokenizer.decode(output_ids[0][len(inputs["input_ids"][0]) :], skip_special_tokens=True))

PagedAttention is a type of attention mechanism that allows for dynamic batching of input data, especially useful when dealing with large datasets or streaming data.

In traditional attention mechanisms like Self-Attention, the model needs to pre-batch all inputs together before applying any attention operation. However, this can lead to inefficiencies and slower training times on large datasets due to repeated computations.

PagedAttention addresses this issue by allowing the model to dynamically batch inputs as they come in, rather than waiting for them all to be processed at once. This means:

1. It can handle very large datasets efficiently.
2. It doesn't require pre-batching.
3. It reduces overhead associated with repeated computation.

The key idea behind PagedAttention is to use multiple layers of attention, each focusing on different parts of the input sequence independently. By doing so, it can effectively utilize parallel processing power while still maintaining the ability to selectively attend to certain parts of the input.

To implement PagedAttention, you typically create a custom attention layer that takes into account the current position (batch index) and the length of the input sequence being handled. The output of the layer is then passed through another set of attention layers, which will focus on different parts of the input based on their positions and lengths.

This approach allows models to process batches of data more efficiently, especially beneficial when working with large-scale datasets where time-to-result becomes an important factor.

In [19]:
with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=2.5,
        #top_k=50,
        #top_p=0.9,
    )


Markdown(tokenizer.decode(output_ids[0][len(inputs["input_ids"][0]) :], skip_special_tokens=True))

PagedAttention in natural language processing, or specifically NLP, stands for Parallel and Delegated Attention with Page-wise Importance.
In the field of neural machine translation (NMT), this technique helps translate longer utterances into shorter, potentially more meaningful outputs. 

1. **Page Level Attention**: This mechanism looks directly for relevant paragraphs that have common parts, then assigns different amounts (from small to large) based on their importance.

    When a word is added:
    - Small attention weights tend toward sentences it's not familiar yet from the sentence above,
      which makes translations easier as you're looking back rather than going forward,
     
2. **Page Breaks** and Pseudo Pages**: To further refine the attention weights so as to focus more towards those paragraphs where they make sense and don't interfere unnecessarily when breaking the main stream passage.

3. This approach aims at providing the best fit at an intermediate layer to the decoder. The input has many layers down before reaching the output tensor; therefore these attention weights are distributed across those multiple attentiveness channels.
So imagine, during each step in processing your long text, we can identify interesting segments and give certain weightings to them while filtering other parts off. These insights come from pages, like "pages" on Wikipedia! This enhances readability by highlighting crucial points while keeping the core message visible. It significantly accelerates processing since you only need to look forward instead if something goes astray in your original message, unlike when focusing entirely around just part of the page that contains your idea. In essence, this strategy reduces redundant noise without necessarily removing everything essential for understanding the overall context, hence making text processing smoother but possibly more obscure due to reduced comprehension accuracy. This method also helps improve coherence because all pieces contribute differently towards achieving coherence, enhancing user experience with a clearer overall presentation compared with single-page summaries.

## Streaming Output

Instead of waiting for full completion,
we can stream tokens as they are generated.

This helps visualize autoregressive decoding.

In [20]:
streamer = TextStreamer(
    tokenizer,
    skip_prompt=True,
    # skip_special_tokens=True
)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        streamer=streamer,
    )

PagedAttention is an attention mechanism designed for handling large amounts of sequential data efficiently, particularly when working with memory constraints or limited computational resources.

Imagine you have a big box of crayons (data) that needs to be organized and displayed on a screen (screen). The problem here is that the box can only hold so many crayons at once before they spill over onto other parts of the screen. This is similar to how our system processes data - we have a lot of information to process, but it's stored in a very small amount of space.

To solve this, PagedAttention uses a technique called "piling up" the data into smaller chunks (or "pages"). Each page contains a subset of the entire dataset, which allows us to use more memory and perform operations faster than if we were to store all the data as one long chunk. This way, we don't waste memory unnecessarily and can still handle large datasets effectively.

The idea behind PagedAttention is to break down th

#### Controlled Streaming with TextIteratorStreamer

Unlike `TextStreamer`, `TextIteratorStreamer` does not print directly.

Instead, it yields generated text chunks that we can:

- Print manually
- Store
- Send to a UI
- Log
- Post-process

In [21]:
from transformers import TextIteratorStreamer
from IPython.display import display, Markdown
import threading

# Create streamer
streamer = TextIteratorStreamer(
    tokenizer,
    skip_prompt=True,
    skip_special_tokens=True
)

generation_kwargs = {
    **inputs,
    "max_new_tokens": max_new_tokens,
    "do_sample": True,
    "streamer": streamer,
}

# Start generation in background
thread = threading.Thread(target=model.generate, kwargs=generation_kwargs)
thread.start()

# Create a display handle (this allows updating)
display_handle = display(Markdown(" # Assistant:\n\n"), display_id=True)

generated_text = ""

for new_text in streamer:
    generated_text += new_text
    
    # Update Markdown output dynamically
    display_handle.update(
        Markdown(f" # Assistant:\n\n{generated_text}")
    )

thread.join()

 # Assistant:

PagedAttention is an attention mechanism designed to handle large amounts of data efficiently, particularly when dealing with very long sequences or large datasets. It's part of the Transformer architecture and is often used in natural language processing (NLP) tasks such as language modeling and text generation.

### Key Components:
1. **Attention Heads**: Each layer in the Transformer model has multiple heads that attend to different parts of the input sequence.
2. **Paging Mechanism**: The attention heads can "page" through the input sequence by skipping certain positions. This helps in maintaining the context during training and improves performance on large datasets.
3. **Batch Size Handling**: The attention mechanism scales well with batch size, making it suitable for handling very long sequences without performance degradation.

### How It Works:
- **Input Sequence**: A sequence of tokens or words from the input dataset.
- **Transformer Model**: The input sequence is fed into the transformer model.
- **Attention Heads**: Multiple attention heads are attached to each token, allowing them to focus on different parts of the input sequence.
- **Page Passing**: Attention heads skip certain positions within the input sequence. These skipped positions represent the gaps where additional information needs to be extracted.
- **Combining Outputs**: The outputs of all attention heads are combined using a softmax function. This gives each token an importance score based on its contribution to the overall attention.
- **Prediction**: The final output is a prediction of the next token in the sequence, taking into account the attention scores of all previous tokens.

### Example:
Suppose we have an input sequence `["apple", "banana", "cherry"]` and we want to predict the next word (`"peach"`).

1. **Transformer Model**:
   - First, the input sequence is passed through two layers of attention heads.
   - After passing through the first layer, the output might look like `[attention_head_0_1, attention_head_1_2, ..., attention_head_6_7]`.
   - After passing through the second layer, the output might look like `[output_0, output_1, ..., output_15]`.

2. **Page Passing**:
   - The first layer outputs `[attention_head_0_1, attention_head_1_2, ..., attention_head_6_7]`. Here, the first 7 elements are skipped because they are already covered by other attention heads.
   - The last element is `output_15`, which contains the attention scores for the entire sequence.

3. **Combining Outputs**:
   - The final output after combining these scores would be something like `[output_15, ...]`, indicating the most probable next word in the sequence.

### Performance Considerations:
- **Batch Size**: The attention mechanism scales well with batch size, so it works effectively even for very long sequences.
- **Training Efficiency**: By avoiding unnecessary computations at the end, it makes training more efficient.
- **Performance**: For large datasets, this method ensures that the model does not overfit to the training set, improving generalization capabilities.

In summary, PagedAttention leverages page passing to manage the complexity of handling extremely long sequences while maintaining good performance.

In [22]:
generated_text

'PagedAttention is an attention mechanism designed to handle large amounts of data efficiently, particularly when dealing with very long sequences or large datasets. It\'s part of the Transformer architecture and is often used in natural language processing (NLP) tasks such as language modeling and text generation.\n\n### Key Components:\n1. **Attention Heads**: Each layer in the Transformer model has multiple heads that attend to different parts of the input sequence.\n2. **Paging Mechanism**: The attention heads can "page" through the input sequence by skipping certain positions. This helps in maintaining the context during training and improves performance on large datasets.\n3. **Batch Size Handling**: The attention mechanism scales well with batch size, making it suitable for handling very long sequences without performance degradation.\n\n### How It Works:\n- **Input Sequence**: A sequence of tokens or words from the input dataset.\n- **Transformer Model**: The input sequence is 

## Summary

In this notebook we learned:

- LLMs operate on token IDs.
- Chat templates structure prompts properly.
- `.generate()` performs autoregressive decoding internally.
- Sampling parameters affect output distribution.
- Streaming reveals token-by-token generation.

In the next notebook, we move to:

→ Batching and inference efficiency.